# Physics-Informed Neural Networks: From Differential Equations to Engineering Intelligence

**Why the future of scientific AI is not about replacing physics — but embedding it directly into machine learning models.**

---

> *This notebook is Part 1 of the Physics AI Series. It is self-contained, requires no prior PINN experience, and every code block is runnable.*

**Target audience:** Chemical engineers, process engineers, pharmaceutical scientists, data scientists entering scientific AI, CFD modelers, graduate students, and AI practitioners interested in NVIDIA Modulus and scientific machine learning.

---

## Table of Contents

1. [The Problem: Why Traditional ML Struggles in Engineering](#section1)
2. [What Is a Physics-Informed Neural Network?](#section2)
3. [The Magic Behind PINNs: Automatic Differentiation](#section3)
4. [Building Our First PINN](#section4)
5. [Why This Is Revolutionary](#section5)
6. [Where PINNs Start Becoming Interesting](#section6)
7. [Limitations of PINNs (Be Honest)](#section7)
8. [Conclusion](#section8)
9. [References](#references)

---
<a id='section1'></a>
## Section 1: The Problem — Why Traditional Machine Learning Struggles in Engineering

### Why does ChatGPT work so well with language?

Large language models are trained on hundreds of billions of examples of human text. The internet is saturated with language data. The statistical patterns in language are **redundant, overlapping, and forgiving** — if the model misses a nuance, the surrounding context usually corrects for it.

Now ask: how much labeled data does a chemical plant generate?

A typical industrial reactor might log temperature, pressure, and concentration at 10-second intervals. Run for a year, that gives you about **3 million data points** — spread across dozens of operating variables, with process upsets, sensor drift, and changing feedstock composition corrupting much of it.

That is not internet-scale data. That is *sparse, expensive, physics-constrained* data.

---

### Engineering data is fundamentally different from internet data

| Property | Language Data | Engineering Data |
|---|---|---|
| Volume | Hundreds of billions of tokens | Thousands to millions of measurements |
| Cost | Near-zero (scraped from web) | Expensive (sensors, experiments, simulations) |
| Redundancy | Extremely high | Low — each measurement is precious |
| Governing structure | Statistical patterns | Physical laws (conservation, thermodynamics) |
| Extrapolation | Risky but often recoverable | Catastrophic if wrong |

---

### Engineering systems are governed by ODEs and PDEs

Almost every physical process in engineering can be written as a differential equation:

**Energy balance in a CSTR:**
$$\rho C_p \frac{dT}{dt} = -\Delta H_r \cdot r(T, C) - \frac{UA}{V}(T - T_c)$$

**Diffusion-reaction in a catalyst pellet:**
$$D_e \nabla^2 C - r(C, T) = 0$$

**Navier-Stokes for incompressible flow:**
$$\rho\left(\frac{\partial \mathbf{u}}{\partial t} + \mathbf{u} \cdot \nabla \mathbf{u}\right) = -\nabla p + \mu \nabla^2 \mathbf{u}$$

These equations encode **centuries of physical understanding**. They tell us exactly how variables relate to each other — not approximately, not statistically, but *precisely*.

---

### What happens when traditional ML ignores physics?

A standard neural network trained only on data will:

1. **Violate conservation laws** — predict a reactor with more energy out than in
2. **Extrapolate poorly** — fail completely outside the training range of temperatures or concentrations
3. **Require enormous labeled datasets** — because it cannot leverage the structure the physics already provides
4. **Produce physically implausible results** — negative concentrations, pressures below vacuum, imaginary velocities

This is not a theoretical concern. It has been observed repeatedly in industrial ML deployments.

---

### The core insight

```
Traditional AI:    Data → Loss → Model

Physics-Informed:  Data + Physical Laws → Loss → Model
```

The physical law is not an afterthought. It is baked into the training objective itself.

That is the idea behind **Physics-Informed Neural Networks**.

---
<a id='section2'></a>
## Section 2: What Is a Physics-Informed Neural Network?

### The difference between an ANN and a PINN

A standard **Artificial Neural Network (ANN)** learns a mapping:
$$f_\theta: x \rightarrow y$$
by minimizing the error between predictions and labeled outputs:
$$\mathcal{L}_{data} = \frac{1}{N}\sum_{i=1}^{N} \left| \hat{y}_i - y_i \right|^2$$

A **Physics-Informed Neural Network (PINN)** learns the same mapping, but adds a second requirement: *whatever the network predicts must also satisfy the governing differential equation.*

$$\mathcal{L}_{total} = \underbrace{\mathcal{L}_{data}}_{\text{fits observations}} + \underbrace{\mathcal{L}_{physics}}_{\text{satisfies the ODE/PDE}} + \underbrace{\mathcal{L}_{BC}}_{\text{satisfies boundary/initial conditions}}$$

---

### A concrete example: the simplest ODE

Consider the first-order linear ODE:
$$\frac{dy}{dx} = -y, \quad y(0) = 1$$

The analytical solution is $y(x) = e^{-x}$.

**Traditional approach:** generate $(x_i, y_i)$ pairs from the true solution, train the network to fit them.

**PINN approach:** 
- Let the network predict $\hat{y}(x)$  
- At a set of *collocation points* $\{x_i\}$, compute the **residual** of the ODE:
$$r(x_i) = \frac{d\hat{y}}{dx}\bigg|_{x_i} + \hat{y}(x_i)$$
- The physics loss forces this residual to zero:
$$\mathcal{L}_{physics} = \frac{1}{N_c}\sum_{i=1}^{N_c} r(x_i)^2$$
- The boundary condition loss enforces $y(0) = 1$:
$$\mathcal{L}_{BC} = \left(\hat{y}(0) - 1\right)^2$$

**No labeled $(x, y)$ pairs are needed.** The differential equation *is* the training signal.

---

### Architecture comparison

```
─────────────────────────────────────────────────────────────
 Traditional Neural Network
─────────────────────────────────────────────────────────────
 Input x  ──▶  [Hidden Layers]  ──▶  ŷ
                                      │
                              Loss = (ŷ - y_true)²

─────────────────────────────────────────────────────────────
 Physics-Informed Neural Network
─────────────────────────────────────────────────────────────
 Input x  ──▶  [Hidden Layers]  ──▶  ŷ
                                      │
                              ┌───────┴────────┐
                              │                │
                     Autograd: dŷ/dx    Boundary check
                              │                │
                     Physics residual    BC residual
                              │                │
                              └───────┬────────┘
                              Total Loss = λ₁·L_data + λ₂·L_phys + λ₃·L_BC
─────────────────────────────────────────────────────────────
```

The key architectural addition is the **automatic differentiation path** that feeds the network's own output derivatives back into the loss. That is the subject of Section 3.

---
<a id='section3'></a>
## Section 3: The Magic Behind PINNs — Automatic Differentiation

### The challenge: how does a neural network compute derivatives?

To enforce $\frac{d\hat{y}}{dx} + \hat{y} = 0$, the network must be able to compute its own derivative with respect to its input $x$.

This sounds circular — but it is precisely what **automatic differentiation (autograd)** provides.

---

### Three ways to differentiate — and why autograd wins

| Method | How | Pros | Cons |
|---|---|---|---|
| **Finite Differences** | $\frac{f(x+h) - f(x)}{h}$ | Simple | Requires mesh, truncation error accumulates |
| **Symbolic Differentiation** | Algebraic rules | Exact | Expression swell, inflexible |
| **Automatic Differentiation** | Chain rule through computation graph | Exact, mesh-free, scalable | Requires AD framework (PyTorch, JAX) |

---

### How PyTorch autograd works

When you compute $\hat{y} = f_\theta(x)$ in PyTorch with `requires_grad=True` on $x$, PyTorch builds a **computational graph** tracking every operation. Calling `.backward()` or `torch.autograd.grad()` traverses this graph in reverse, applying the chain rule exactly — to machine precision.

This means:
- $\frac{d\hat{y}}{dx}$ is available for free, no mesh required
- $\frac{d^2\hat{y}}{dx^2}$ requires a second pass — also free
- $\frac{\partial^2 u}{\partial x^2} + \frac{\partial^2 u}{\partial y^2}$ (the Laplacian) is computable for any 2D network

Let's see this in action.

In [ ]:
# Install dependencies if needed
# !pip install torch numpy matplotlib

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# ─── Demonstrate automatic differentiation ───────────────────────────────────
# Let's differentiate a simple function: f(x) = x³ + 2x² - 5x + 1
# Analytical: f'(x) = 3x² + 4x - 5

x = torch.tensor([2.0], requires_grad=True)   # x = 2, track gradient

f = x**3 + 2*x**2 - 5*x + 1

# Compute df/dx via autograd
dfdx = torch.autograd.grad(f, x, create_graph=True)[0]

# Compute d²f/dx² via autograd (second derivative)
d2fdx2 = torch.autograd.grad(dfdx, x)[0]

print(f"x = {x.item():.1f}")
print(f"f(x)     = {f.item():.4f}  (expected: {2**3 + 2*2**2 - 5*2 + 1:.4f})")
print(f"f'(x)    = {dfdx.item():.4f}  (expected: {3*2**2 + 4*2 - 5:.4f})")
print(f"f''(x)   = {d2fdx2.item():.4f}  (expected: {6*2 + 4:.4f})")

Autograd computes derivatives **exactly** — no finite difference approximation, no mesh. This is the foundation that makes PINNs possible.

When the network $f_\theta(x)$ replaces the simple polynomial above, autograd can still compute $\frac{d f_\theta}{dx}$ through all the layers, activation functions, and weights — automatically.

---
<a id='section4'></a>
## Section 4: Building Our First PINN

### Problem statement

$$\frac{dy}{dx} = -y, \quad y(0) = 1, \quad x \in [0, 5]$$

**Analytical solution:** $y(x) = e^{-x}$

We will solve this using a PINN with **zero labeled data** — the differential equation and boundary condition are the only training signals.

### Notebook objectives
By the end of this section you will be able to:
- Define a neural network as a function approximator
- Define a PDE residual loss
- Define a boundary condition loss
- Train the PINN
- Compare the result with the analytical solution

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# ─── Step 1: Define the neural network ───────────────────────────────────────
# A fully-connected network: input=1 (x), output=1 (y_hat)
# Tanh activations work well for PINNs because they are infinitely differentiable

class PINN(nn.Module):
    def __init__(self, layers):
        super().__init__()
        network = []
        for i in range(len(layers) - 1):
            network.append(nn.Linear(layers[i], layers[i+1]))
            if i < len(layers) - 2:
                network.append(nn.Tanh())
        self.net = nn.Sequential(*network)
        self._init_weights()

    def _init_weights(self):
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)


# Architecture: 1 → 32 → 32 → 32 → 1
model = PINN([1, 32, 32, 32, 1]).to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters())}")

In [ ]:
# ─── Step 2: Define the domain and collocation points ─────────────────────────
# Collocation points: where we enforce the ODE residual
# No labels needed — just x values scattered across [0, 5]

N_collocation = 1000   # points where ODE residual is enforced

x_col = torch.linspace(0, 5, N_collocation).view(-1, 1).to(device)
x_col.requires_grad_(True)   # must track gradient to differentiate w.r.t. x

# Boundary / initial condition point
x_bc = torch.tensor([[0.0]]).to(device)   # x = 0
y_bc = torch.tensor([[1.0]]).to(device)   # y(0) = 1

print(f"Collocation points: {x_col.shape}")
print(f"Boundary point:     x={x_bc.item()}, y={y_bc.item()}")

In [ ]:
# ─── Step 3: Define the physics residual ─────────────────────────────────────
# ODE: dy/dx + y = 0
# Residual r(x) = dy_hat/dx + y_hat
# Perfect solution ⟹ r(x) = 0 everywhere

def ode_residual(model, x):
    y_hat = model(x)
    # Compute dy_hat/dx via autograd
    dy_dx = torch.autograd.grad(
        y_hat, x,
        grad_outputs=torch.ones_like(y_hat),
        create_graph=True   # needed so this loss is differentiable w.r.t. weights
    )[0]
    residual = dy_dx + y_hat   # should equal zero
    return residual

In [ ]:
# ─── Step 4: Define the combined loss ─────────────────────────────────────────

def total_loss(model, x_col, x_bc, y_bc):
    # Physics loss: ODE residual at collocation points
    r = ode_residual(model, x_col)
    loss_physics = torch.mean(r**2)

    # Boundary condition loss: y(0) = 1
    y_bc_pred = model(x_bc)
    loss_bc = torch.mean((y_bc_pred - y_bc)**2)

    # Equal weighting; can be tuned
    return loss_physics + loss_bc, loss_physics.item(), loss_bc.item()

In [ ]:
# ─── Step 5: Train the PINN ───────────────────────────────────────────────────

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

N_epochs = 5000
history = {'total': [], 'physics': [], 'bc': []}

for epoch in range(N_epochs):
    optimizer.zero_grad()
    loss, l_phys, l_bc = total_loss(model, x_col, x_bc, y_bc)
    loss.backward()
    optimizer.step()

    history['total'].append(loss.item())
    history['physics'].append(l_phys)
    history['bc'].append(l_bc)

    if epoch % 500 == 0:
        print(f"Epoch {epoch:5d} | Total: {loss.item():.2e} | "
              f"Physics: {l_phys:.2e} | BC: {l_bc:.2e}")

print("\nTraining complete.")

In [ ]:
# ─── Step 6: Plot the results ─────────────────────────────────────────────────

model.eval()
x_test = torch.linspace(0, 5, 500).view(-1, 1).to(device)

with torch.no_grad():
    y_pred = model(x_test).cpu().numpy()

x_np = x_test.cpu().numpy().flatten()
y_analytical = np.exp(-x_np)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: PINN vs analytical solution
axes[0].plot(x_np, y_analytical, 'k-', linewidth=2.5, label='Analytical: $y = e^{-x}$')
axes[0].plot(x_np, y_pred, 'r--', linewidth=2, label='PINN prediction')
axes[0].set_xlabel('x', fontsize=13)
axes[0].set_ylabel('y', fontsize=13)
axes[0].set_title('PINN vs Analytical Solution\n$dy/dx = -y,\ y(0)=1$', fontsize=13)
axes[0].legend(fontsize=12)
axes[0].grid(True, alpha=0.3)

# Right: Training loss
epochs = np.arange(len(history['total']))
axes[1].semilogy(epochs, history['total'],  'b-',  linewidth=1.5, label='Total loss')
axes[1].semilogy(epochs, history['physics'], 'g--', linewidth=1.5, label='Physics loss')
axes[1].semilogy(epochs, history['bc'],      'r:',  linewidth=2,   label='BC loss')
axes[1].set_xlabel('Epoch', fontsize=13)
axes[1].set_ylabel('Loss (log scale)', fontsize=13)
axes[1].set_title('Training History', fontsize=13)
axes[1].legend(fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('pinn_first_example.png', dpi=150, bbox_inches='tight')
plt.show()

# Error statistics
abs_error = np.abs(y_pred.flatten() - y_analytical)
print(f"Max absolute error:  {abs_error.max():.4e}")
print(f"Mean absolute error: {abs_error.mean():.4e}")

### What just happened?

The PINN solved a differential equation **without any labeled $(x, y)$ pairs**. The training signal was entirely:
1. The requirement that $\frac{d\hat{y}}{dx} + \hat{y} = 0$ at 1000 collocation points
2. The single initial condition $y(0) = 1$

This is the core idea of PINNs — the differential equation provides information that replaces the need for labeled data.

---

### Bonus: PINN with sparse observational data

In many engineering problems you have *some* measurements but not many. Let's see how adding just 10 sparse observations improves the PINN.

In [ ]:
# ─── PINN with sparse data + physics ─────────────────────────────────────────
torch.manual_seed(0)

model2 = PINN([1, 32, 32, 32, 1]).to(device)
optimizer2 = torch.optim.Adam(model2.parameters(), lr=1e-3)

# Sparse observations: 10 noisy measurements
N_obs = 10
x_obs_np = np.linspace(0, 5, N_obs)
y_obs_np  = np.exp(-x_obs_np) + 0.02 * np.random.randn(N_obs)   # 2% noise
x_obs = torch.tensor(x_obs_np, dtype=torch.float32).view(-1,1).to(device)
y_obs = torch.tensor(y_obs_np, dtype=torch.float32).view(-1,1).to(device)

for epoch in range(5000):
    optimizer2.zero_grad()
    # Physics loss
    r = ode_residual(model2, x_col)
    l_phys = torch.mean(r**2)
    # BC loss
    l_bc = (model2(x_bc) - y_bc)**2
    # Data loss (sparse observations)
    l_data = torch.mean((model2(x_obs) - y_obs)**2)
    # Combined
    loss2 = l_phys + l_bc + l_data
    loss2.backward()
    optimizer2.step()

model2.eval()
with torch.no_grad():
    y_pred2 = model2(x_test).cpu().numpy()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_np, y_analytical, 'k-', linewidth=2.5, label='Analytical')
ax.plot(x_np, y_pred,  'r--', linewidth=2,   label='PINN (no data)')
ax.plot(x_np, y_pred2, 'b-',  linewidth=2,   label='PINN (physics + 10 noisy obs.)')
ax.scatter(x_obs_np, y_obs_np, c='blue', zorder=5, s=60, label='Noisy observations', edgecolors='k')
ax.set_xlabel('x', fontsize=13)
ax.set_ylabel('y', fontsize=13)
ax.set_title('Physics-Only vs Physics + Sparse Data', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('pinn_with_sparse_data.png', dpi=150, bbox_inches='tight')
plt.show()

---
<a id='section5'></a>
## Section 5: Why This Is Revolutionary

### Comparing classical solvers vs PINNs

| Method | Requires Mesh? | Handles Inverse Problems? | Works with Sparse Data? | Continuous Solution? |
|---|:---:|:---:|:---:|:---:|
| Euler Method | Yes | No | No | No |
| Runge-Kutta 4 | Yes | No | No | No |
| Finite Element | Yes | Hard | No | No |
| Finite Difference | Yes | Hard | No | No |
| **PINN** | **No** | **Yes** | **Yes** | **Yes** |

The "Continuous Solution" column deserves emphasis. A PINN is a **differentiable function** defined everywhere on the domain — not just at grid points. You can query it at any $x$ value and get a smooth prediction.

---

### Key advantages

**1. Mesh-free**  
Traditional PDE solvers discretize space and time into meshes. Creating a good mesh for a complex 3D geometry (a pharmaceutical tablet, a turbine blade, a reactor with baffles) can take days of engineering effort. PINNs learn a continuous function over the domain directly — no mesh required.

**2. Inverse problems become straightforward**  
In a classical solver, if you don't know a parameter (e.g., a reaction rate constant), you must solve an optimization problem wrapped around the solver — expensive and brittle. In a PINN, unknown parameters become additional trainable variables in the same training loop.

**3. Works with sparse and noisy data**  
As shown above, PINNs can assimilate experimental measurements directly through the data loss term — the physics automatically regularizes the solution between measurement points.

**4. Continuous and differentiable everywhere**  
The PINN outputs a continuous function. You can compute gradients, Hessians, and integral quantities analytically over the entire domain.

---

### Visualizing the advantage: mesh-free solution

In [ ]:
# ─── Visualize PINN as a continuous function ─────────────────────────────────
# Query the PINN at any resolution — it doesn't need to be retrained

model.eval()
resolutions = [10, 50, 200, 1000]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, N in zip(axes, resolutions):
    x_query = torch.linspace(0, 5, N).view(-1,1).to(device)
    with torch.no_grad():
        y_query = model(x_query).cpu().numpy()
    x_q_np  = x_query.cpu().numpy().flatten()

    ax.plot(x_q_np, np.exp(-x_q_np), 'k-', lw=2, label='Analytical')
    ax.plot(x_q_np, y_query, 'r--', lw=1.5, label=f'PINN ({N} pts)')
    ax.set_title(f'{N} query points', fontsize=11)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('PINN queried at different resolutions — no retraining needed', fontsize=12)
plt.tight_layout()
plt.savefig('pinn_continuous_solution.png', dpi=150, bbox_inches='tight')
plt.show()

---
<a id='section6'></a>
## Section 6: Where PINNs Start Becoming Interesting

The first-order ODE $dy/dx = -y$ is a proof of concept. Here is a preview of where this framework leads:

---

### Article 2 — Second-Order ODEs: Oscillators and Spring-Mass Systems

$$m\frac{d^2x}{dt^2} + c\frac{dx}{dt} + kx = F(t)$$

PINNs handle second derivatives as naturally as first — autograd computes them in one extra pass. The spring-mass system introduces damping, resonance, and superposition — all important in process equipment vibration analysis.

---

### Article 3 — Heat Transfer Equation

$$\frac{\partial T}{\partial t} = \alpha \frac{\partial^2 T}{\partial x^2}$$

The 1D heat equation is the gateway to 2D/3D heat transfer. PINNs solve it without meshing — critical for complex geometries like pharmaceutical tablet dissolution or heat exchanger fouling.

---

### Article 4 — Reaction Kinetics: $A \rightarrow B$

$$\frac{dC_A}{dt} = -kC_A, \quad \frac{dC_B}{dt} = kC_A$$

The simplest kinetic system — directly applicable to pharmaceutical manufacturing, where reaction rate and conversion are critical process parameters.

---

### Article 5 — Parameter Estimation: Learning Rate Constants from Data

Given noisy concentration measurements $C_A(t_{exp})$, estimate the unknown rate constant $k$. With a PINN, $k$ becomes an additional trainable parameter — no separate optimization loop needed.

$$\mathcal{L} = \underbrace{\|C_A^{pred} - C_A^{exp}\|^2}_{\text{fit data}} + \underbrace{\|\dot{C}_A + kC_A\|^2}_{\text{satisfy kinetics}}$$

---

### Article 6 — Navier-Stokes: The Entry Point into CFD

$$\nabla \cdot \mathbf{u} = 0, \quad \rho(\mathbf{u} \cdot \nabla)\mathbf{u} = -\nabla p + \mu \nabla^2 \mathbf{u}$$

Raissi et al. (2019) demonstrated that PINNs can infer hidden velocity fields and pressure from sparse flow observations — a capability that would require weeks of CFD setup with traditional methods.

In [ ]:
# ─── Preview: Reaction kinetics PINN (A → B) ─────────────────────────────────
# A brief teaser showing PINNs on a 2-species system
# Will be developed fully in Article 4

torch.manual_seed(7)

class ReactionPINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 2)   # outputs: [C_A, C_B]
        )
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, t):
        return self.net(t)

k_true = 2.0   # true rate constant (we'll pretend we know it here)

rxn_model = ReactionPINN().to(device)
rxn_opt   = torch.optim.Adam(rxn_model.parameters(), lr=1e-3)

t_col = torch.linspace(0, 3, 500).view(-1,1).to(device)
t_col.requires_grad_(True)

t_ic = torch.tensor([[0.0]]).to(device)
ic   = torch.tensor([[1.0, 0.0]]).to(device)   # C_A(0)=1, C_B(0)=0

for epoch in range(6000):
    rxn_opt.zero_grad()
    C = rxn_model(t_col)          # [N, 2]: columns are C_A, C_B
    C_A, C_B = C[:, 0:1], C[:, 1:2]

    dCA_dt = torch.autograd.grad(C_A, t_col, torch.ones_like(C_A), create_graph=True)[0]
    dCB_dt = torch.autograd.grad(C_B, t_col, torch.ones_like(C_B), create_graph=True)[0]

    res_A = dCA_dt + k_true * C_A
    res_B = dCB_dt - k_true * C_A

    l_phys = torch.mean(res_A**2) + torch.mean(res_B**2)
    l_ic   = torch.mean((rxn_model(t_ic) - ic)**2)

    loss = l_phys + l_ic
    loss.backward()
    rxn_opt.step()

rxn_model.eval()
t_test = torch.linspace(0, 3, 300).view(-1,1).to(device)
with torch.no_grad():
    C_pred = rxn_model(t_test).cpu().numpy()

t_np = t_test.cpu().numpy().flatten()
CA_true = np.exp(-k_true * t_np)
CB_true = 1 - CA_true

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(t_np, CA_true, 'b-',  lw=2.5, label='$C_A$ analytical')
ax.plot(t_np, CB_true, 'r-',  lw=2.5, label='$C_B$ analytical')
ax.plot(t_np, C_pred[:,0], 'b--', lw=2, label='$C_A$ PINN')
ax.plot(t_np, C_pred[:,1], 'r--', lw=2, label='$C_B$ PINN')
ax.set_xlabel('Time', fontsize=13)
ax.set_ylabel('Concentration', fontsize=13)
ax.set_title('Preview: PINN for $A \\rightarrow B$ Reaction Kinetics (k=2.0)', fontsize=13)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('pinn_reaction_preview.png', dpi=150, bbox_inches='tight')
plt.show()

---
<a id='section7'></a>
## Section 7: Limitations of PINNs (Be Honest)

Most introductory articles stop after the success stories. We should not.

PINNs are a genuinely promising technique with real, well-documented failure modes. Understanding them is essential for anyone planning to deploy PINNs in industrial applications.

---

### 1. Training instability

The PINN loss landscape is non-convex and poorly conditioned. The gradients flowing from the physics loss and the boundary condition loss can point in conflicting directions, causing oscillations or divergence during training.

**Symptom:** Training loss oscillates or plateaus far above zero even with a well-posed problem.  
**Mitigation:** Adaptive loss weighting, learning rate scheduling, residual-based adaptive refinement (RAR) [Lu et al., 2021].

---

### 2. Stiff ODEs and PDEs

A stiff system is one where different components evolve at vastly different timescales. An example from chemical engineering:

$$\frac{dC}{dt} = -10^6 C, \quad C(0) = 1$$

This decays to near-zero within microseconds. Standard PINN training with uniform collocation points completely misses the dynamics.

**Mitigation:** Adaptive collocation, curriculum training (gradually expanding the time horizon), domain decomposition.

---

### 3. Slow convergence

PINNs typically require tens of thousands of epochs to converge — orders of magnitude more iterations than classical solvers for equivalent accuracy.

**Quantitative comparison:**

| Approach | Time to solve 1D heat equation |
|---|---|
| Finite difference (100 grid pts) | < 1 ms |
| PINN (5000 epochs, GPU) | ~ 30–60 s |

PINNs are not faster solvers. Their advantage lies in other dimensions: inverse problems, sparse data, complex geometries.

---

### 4. Scaling to high dimensions

PINNs struggle with long time horizons and high-dimensional PDEs. A 3D Navier-Stokes simulation with complex boundary conditions at high Reynolds numbers remains largely unsolved by PINNs.

**Mitigation:** Extended PINNs (XPINNs) for domain decomposition, Fourier Neural Operators (FNO) for operator learning.

---

### 5. Multi-physics coupling

When multiple coupled equations are present (heat + mass + momentum + reaction), the relative scaling of residuals becomes critical. A poorly scaled system causes one physics term to dominate the loss and the others to be ignored.

**Mitigation:** Non-dimensionalization, learned loss weights, multi-task learning strategies.

---

### Beyond vanilla PINNs

| Method | Addresses |
|---|---|
| **XPINNs** | Long time horizons, stiff systems via domain decomposition |
| **Fourier Neural Operators (FNO)** | Amortized solution of PDE families (one model, any BC) |
| **DeepONet** | Operator learning — maps input functions to output functions |
| **Neural Operators** | General operator learning for PDEs on irregular domains |
| **PirateNets** | Training instability via adaptive residual connections |

Each of these is a potential future article.

In [ ]:
# ─── Demonstrate training instability: effect of collocation density ──────────
torch.manual_seed(1)

results = {}
for N in [10, 100, 1000]:
    m = PINN([1, 32, 32, 32, 1]).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    xc = torch.linspace(0, 5, N).view(-1,1).to(device)
    xc.requires_grad_(True)
    losses = []
    for _ in range(3000):
        opt.zero_grad()
        r = ode_residual(m, xc)
        l_p = torch.mean(r**2)
        l_b = (m(x_bc) - y_bc)**2
        loss = l_p + l_b
        loss.backward()
        opt.step()
        losses.append(loss.item())
    results[N] = losses

fig, ax = plt.subplots(figsize=(9, 5))
for N, losses in results.items():
    ax.semilogy(losses, label=f'{N} collocation points')
ax.set_xlabel('Epoch', fontsize=13)
ax.set_ylabel('Total Loss', fontsize=13)
ax.set_title('Effect of Collocation Point Density on Convergence', fontsize=13)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('pinn_collocation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
<a id='section8'></a>
## Section 8: Conclusion

The important idea is not that neural networks *replace* physics.

The important idea is that physics becomes *part of* the neural network.

For decades, engineers described systems through equations. For the last decade, AI described systems through data. **PINNs represent the convergence of these two worlds.**

---

### What we built in this notebook

1. Defined a neural network as a function approximator over a continuous domain
2. Used autograd to compute exact derivatives of the network output with respect to inputs
3. Enforced an ODE residual as a training loss — with zero labeled data
4. Combined physics loss with sparse noisy observations
5. Previewed the reaction kinetics system $A \rightarrow B$
6. Honestly assessed where PINNs currently struggle

---

### What's next in this series

| Article | Topic | Complexity |
|---|---|---|
| **2** | Second-order ODEs: oscillators | ⬛⬜⬜⬜ |
| **3** | Heat transfer equation (1D PDE) | ⬛⬛⬜⬜ |
| **4** | Reaction kinetics: $A \rightarrow B \rightarrow C$ | ⬛⬛⬜⬜ |
| **5** | Parameter estimation: learning $k$ from noisy data | ⬛⬛⬛⬜ |
| **6** | Navier-Stokes: entry point to CFD | ⬛⬛⬛⬛ |

---

### The convergence

This convergence is already transforming:
- **Pharmaceutical manufacturing**: Real-time parameter estimation from PAT data
- **Computational fluid dynamics**: Mesh-free solvers for complex geometries  
- **Digital twins**: Physics-constrained surrogate models that generalize beyond training data
- **Reaction engineering**: Simultaneous identification of kinetics and transport from sparse pilot data
- **Scientific discovery**: Identifying governing equations from experimental observations

The tools to participate in this transformation are now open-source, well-documented, and runnable on a standard laptop. This notebook is your starting point.

---
<a id='references'></a>
## References

All PDFs are available in the `references/` folder.

---

### Foundational Papers

[1] **Raissi, M., Perdikaris, P., & Karniadakis, G. E.** (2019).  
*Physics-informed neural networks: A deep learning framework for solving forward and inverse problems involving nonlinear partial differential equations.*  
Journal of Computational Physics, 378, 686–707.  
DOI: [10.1016/j.jcp.2018.10.045](https://doi.org/10.1016/j.jcp.2018.10.045)  
arXiv: [1711.10561](https://arxiv.org/abs/1711.10561)  
📄 `references/Raissi2019_PINN_JCP.pdf`

[2] **Karniadakis, G. E., Kevrekidis, I. G., Lu, L., Perdikaris, P., Wang, S., & Yang, L.** (2021).  
*Physics-informed machine learning.*  
Nature Reviews Physics, 3, 422–440.  
DOI: [10.1038/s42254-021-00314-5](https://doi.org/10.1038/s42254-021-00314-5)  
📄 `references/Karniadakis2021_NatureReviewsPhysics.pdf`

---

### Software and Libraries

[3] **Lu, L., Meng, X., Mao, Z., & Karniadakis, G. E.** (2021).  
*DeepXDE: A deep learning library for solving differential equations.*  
SIAM Review, 63(1), 208–228.  
DOI: [10.1137/19M1274067](https://doi.org/10.1137/19M1274067)  
arXiv: [1907.04502](https://arxiv.org/abs/1907.04502)  
📄 `references/Lu2021_DeepXDE_SIAMREVIEW.pdf`

---

### Review Papers

[4] **Cuomo, S., di Cola, V. S., Giampaolo, F., Rozza, G., Raissi, M., & Piccialli, F.** (2022).  
*Scientific machine learning through physics-informed neural networks: Where we are and what's next.*  
Journal of Scientific Computing, 92(88).  
DOI: [10.1007/s10915-022-01939-z](https://doi.org/10.1007/s10915-022-01939-z)  
arXiv: [2201.05624](https://arxiv.org/abs/2201.05624)  
📄 `references/Cuomo2022_SciML_Review.pdf`

[5] **Cai, S., Mao, Z., Wang, Z., Yin, M., & Karniadakis, G. E.** (2021).  
*Physics-informed neural networks (PINNs) for fluid mechanics: A review.*  
Acta Mechanica Sinica, 37, 1727–1738.  
arXiv: [2105.09506](https://arxiv.org/abs/2105.09506)  
📄 `references/Cai2021_PINNs_FluidMechanics.pdf`

---

### Recent Advances (2024–2026)

[6] **Wang, S., Li, B., Chen, Y., & Perdikaris, P.** (2024).  
*PirateNets: Physics-informed deep learning with residual adaptive networks.*  
arXiv: [2402.00326](https://arxiv.org/abs/2402.00326)  
📄 `references/Wang2024_PirateNets.pdf`

[7] **Hu, Z., Jagtap, A. D., Karniadakis, G. E., & Kawaguchi, K.** (2023).  
*Augmented physics-informed neural networks (APINNs): A gating network-based soft domain decomposition methodology.*  
Engineering Applications of Artificial Intelligence, 126, 107183.  
DOI: [10.1016/j.engappai.2023.107183](https://doi.org/10.1016/j.engappai.2023.107183)  
arXiv: [2211.08939](https://arxiv.org/abs/2211.08939)

---

### Documentation and Tutorials

[8] **PyTorch Autograd documentation.**  
https://pytorch.org/docs/stable/autograd.html

[9] **DeepXDE documentation and examples.**  
https://deepxde.readthedocs.io

[10] **NVIDIA Modulus documentation.**  
https://docs.nvidia.com/modulus

---

*Physics AI Series — Article 1 of N*  
*Author: Ali Shahmohammadi*